# buffer-copy_-inplace composite — cx7: copy_ velocity buffer, then in-place SGD step on the param

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `buffer-copy_-inplace`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "buffer-copy_-inplace"
DD_ATOM_IDS = ["buffer-copy_-inplace", "inplace-param-update"]
DD_SUBTOPICS = ["PyTorch: in-place buffer copy", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Inside a hand-rolled SGD-momentum optimizer, ONE step does two in-place mutations on tensors owned by the optimizer:

1. **`buffer-copy_-inplace`** — the velocity buffer `v` stored in `state[p]` is updated without rebinding: `v.copy_(new_v)`. If you write `v = new_v` instead, you only rebind the local variable; the entry in `state[p]['momentum_buffer']` still points to the old tensor.
2. **`inplace-param-update`** — the parameter `p` is mutated via `p.data.add_(v, alpha=-lr)` (or equivalently `p.data -= lr * v`). Mutating `p.data` (not `p` itself) avoids touching the autograd graph; the param storage IS updated so the next forward pass sees new weights.

**Anatomy.**
```python
for p in params:
    g = p.grad
    v = state[p]['momentum_buffer']
    new_v = mu * v + g
    v.copy_(new_v)                       # buffer-copy_-inplace
    p.data.add_(v, alpha=-lr)            # inplace-param-update
```

**Why both atoms together.** A correct buffer update is wasted if the param isn't actually mutated; a correct param update is wasted if the buffer was rebound and reset to the prior value next step. They both have to be IN-PLACE for the optimizer to be stateful.

### Composite Exercise — copy_ velocity buffer, then in-place SGD step on the param

**Atoms exercised together**: `buffer-copy_-inplace`, `inplace-param-update`

Implement `cx7_sgd_momentum_step(params, grads, velocity_buffers, lr, mu)`.

For each `(p, g, v)` triple drawn from the three input lists, run one step of SGD with momentum so that:

1. The new velocity `mu * v + g` is written INTO the existing buffer tensor `v` via `v.copy_(...)` — DO NOT rebind. The caller still holds the same tensor object in `velocity_buffers[i]` and expects it to carry the new value.
2. The parameter `p` is updated in place: `p.data.add_(v, alpha=-lr)` (or the equivalent `p.data -= lr * v`). The test verifies `p`'s storage pointer is unchanged.

Return `None`. The mutations are the whole point.

The test runs TWO consecutive steps with the same buffers/params and cross-checks against `torch.optim.SGD(momentum=...)`. If either atom is missing — buffer rebind, or param rebind — step 2 will diverge from the PyTorch reference.

In [ ]:
def cx7_sgd_momentum_step(params, grads, velocity_buffers, lr, mu):
    for p, g, v in zip(params, grads, velocity_buffers):
        # Atom A (buffer-copy_-inplace): write the new velocity INTO v, no rebind.
        v.copy_(mu * v + g)
        # Atom B (inplace-param-update): mutate p.data so storage is unchanged.
        p.data.add_(v, alpha=-lr)
    return None


<details><summary>Show solution — cx7</summary>

```python
def cx7_sgd_momentum_step(params, grads, velocity_buffers, lr, mu):
    for p, g, v in zip(params, grads, velocity_buffers):
        # Atom A (buffer-copy_-inplace): write the new velocity INTO v, no rebind.
        v.copy_(mu * v + g)
        # Atom B (inplace-param-update): mutate p.data so storage is unchanged.
        p.data.add_(v, alpha=-lr)
    return None
```

Two failure modes the test catches:
- `v = mu * v + g` (rebind) — step 1 looks right because the formula is right, but step 2 diverges from `torch.optim.SGD` because `velocity_buffers[i]` still points at the original zero tensor.
- `p = p - lr * v` (param rebind) — caller still holds the OLD parameter; their forward pass sees stale weights.
`p.data.add_(v, alpha=-lr)` is the literal in-place op. `p.data -= lr * v` works too but allocates a temporary; `add_` doesn't.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["PyTorch: in-place buffer copy", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()